In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# Charger le fichier de données nettoyé
# Assurez-vous d'utiliser le bon chemin relatif par rapport à votre dossier 'data'
df_clean = pd.read_csv('../data_DVF_clean.csv')

print(f"✅ Données propres chargées. {len(df_clean):,} transactions disponibles pour l'analyse.")
display(df_clean.head())

✅ Données propres chargées. 1,816,784 transactions disponibles pour l'analyse.


,date_mutation,valeur_fonciere,commune,surface_reelle_bati,code_postal,prix_au_m2,type_de_bien
0,2020-07-03,179970.0,SAINT-LAURENT-SUR-SAONE,61.0,1750.0,2950.327869,T3
1,2020-07-02,136000.0,BOURG-EN-BRESSE,62.0,1000.0,2193.548387,T3
2,2020-07-06,124000.0,MONTREVEL-EN-BRESSE,53.0,1340.0,2339.622642,T2
3,2020-07-01,270000.0,BOURG-EN-BRESSE,111.0,1000.0,2432.432432,T2
4,2020-07-01,270000.0,BOURG-EN-BRESSE,31.0,1000.0,8709.677419,T2


In [43]:
def afficher_classement_villes(type_bien_selectionne='T2', nb_villes_a_afficher=10):
    """
    Calcule et affiche les prix moyens au m² pour les villes selon le type de bien.
    """
    
    # 1. Filtrer les données selon le type de bien sélectionné
    df_filtre = df_clean[df_clean['type_de_bien'] == type_bien_selectionne].copy()
    
    if df_filtre.empty:
        print(f"Aucune donnée trouvée pour le type de bien : {type_bien_selectionne}.")
        return

    # 2. Calculer le prix moyen au m² par commune (en excluant les petites communes avec peu de transactions)
    df_classement = df_filtre.groupby('commune')['prix_au_m2'].agg(['mean', 'count']).reset_index()
    
    # Filtrer les communes avec moins de 50 transactions pour avoir une moyenne fiable
    df_classement = df_classement[df_classement['count'] >= 50]
    df_classement = df_classement.sort_values(by='mean', ascending=False)
    df_classement['mean'] = df_classement['mean'].round(0).astype(int) # Arrondir pour la lisibilité
    
    # 3. Séparer le classement en 'Plus Chères' et 'Moins Chères'
    df_plus_cheres = df_classement.head(nb_villes_a_afficher)
    df_moins_cheres = df_classement.tail(nb_villes_a_afficher).sort_values(by='mean', ascending=True)

    # 4. Affichage (Matplotlib)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'Classement du Prix Moyen au m² pour les biens de type {type_bien_selectionne}', fontsize=16)

    # Graphe 1: Villes les plus chères (Zones à éviter ou trop chères)
    axes[0].barh(df_plus_cheres['commune'], df_plus_cheres['mean'], color='salmon')
    axes[0].set_title(f'{nb_villes_a_afficher} Villes les Plus Chères (Prix élevé ⬆️)')
    axes[0].set_xlabel('Prix moyen au m² (€)')
    axes[0].invert_yaxis()
    
    # Graphe 2: Villes les moins chères (Zones à potentiel)
    axes[1].barh(df_moins_cheres['commune'], df_moins_cheres['mean'], color='lightgreen')
    axes[1].set_title(f'{nb_villes_a_afficher} Villes les Moins Chères (Potentiel d\'investissement ⬇️)')
    axes[1].set_xlabel('Prix moyen au m² (€)')
    axes[1].invert_yaxis()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
# Ajout d'une balise pour un diagramme conceptuel

In [44]:
# Définir les options pour le menu déroulant
options_type = sorted(df_clean['type_de_bien'].unique().tolist())

# Créer le widget (menu déroulant)
type_bien_select = widgets.Dropdown(
    options=options_type,
    value='T2',  # Valeur par défaut
    description='Type de bien :',
    disabled=False,
)

# Créer le widget (curseur) pour le nombre de villes
nb_villes_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description='Nombre de villes :',
    disabled=False,
    continuous_update=False
)

# Lier les widgets à la fonction
interactive_classement = widgets.interactive(
    afficher_classement_villes,
    type_bien_selectionne=type_bien_select,
    nb_villes_a_afficher=nb_villes_slider
)

# Afficher l'ensemble
display(Markdown('### 🏙️ Outil 1 : Prix Moyen au M² par Localisation (Besoin n°2)'))
display(Markdown('**Instruction pour Hugo :** Utilisez les filtres ci-dessous pour trouver rapidement les villes où votre budget (environ 4000 €/m² pour un T2 à 200k€) est le plus réaliste.'))
display(interactive_classement)

### 🏙️ Outil 1 : Prix Moyen au M² par Localisation (Besoin n°2)

**Instruction pour Hugo :** Utilisez les filtres ci-dessous pour trouver rapidement les villes où votre budget (environ 4000 €/m² pour un T2 à 200k€) est le plus réaliste.

interactive(children=(Dropdown(description='Type de bien :', index=1, options=('T1', 'T2', 'T3'), value='T2'),…

In [45]:
# --- 3. Création et Affichage des Widgets d'Entrée ---

# Liste des villes et types de biens disponibles dans le dataset filtré
villes_disponibles = sorted(df_prix_moyen['commune'].unique().tolist())
types_disponibles = sorted(df_prix_moyen['type_de_bien'].unique().tolist())

# --- CORRECTION ICI : Définir la valeur par défaut ---
# On utilise la première ville de la liste comme valeur par défaut,
# car elle est garantie d'exister dans la liste 'villes_disponibles'.
valeur_ville_defaut = villes_disponibles[0] if villes_disponibles else None

# Gérer la valeur par défaut pour le type de bien (T2 est un bon pari)
valeur_type_defaut = 'T2' if 'T2' in types_disponibles else types_disponibles[0] if types_disponibles else None
# ----------------------------------------------------

# WIDGETS
# On utilise la variable 'valeur_ville_defaut' au lieu du nom 'Lyon' écrit en dur
ville_select = widgets.Dropdown(options=villes_disponibles, value=valeur_ville_defaut, description='Ville :')
type_bien_select = widgets.Dropdown(options=types_disponibles, value=valeur_type_defaut, description='Type de bien :')
surface_slider = widgets.IntSlider(value=40, min=15, max=80, step=5, description='Surface (m²) :')

# Paramètres de simulation (Hypothèses d'Hugo)
prix_nuit_input = widgets.FloatSlider(value=70.0, min=30.0, max=150.0, step=5.0, description='Prix/Nuit (€) :')
taux_occupation_input = widgets.IntSlider(value=60, min=20, max=95, step=5, description='Taux Occupation (%) :')
charges_pct_input = widgets.FloatSlider(value=15.0, min=5.0, max=30.0, step=1.0, description='Charges (% Revenus) :')

# Liaison des widgets à la fonction, ce qui CRÉE la variable 'interactive_simulation'
interactive_simulation = widgets.interactive(
    simuler_et_recommander,
    ville=ville_select,
    type_bien=type_bien_select,
    surface=surface_slider,
    prix_nuit_estime=prix_nuit_input,
    taux_occupation_cible=taux_occupation_input,
    charges_annuelles_pct=charges_pct_input
)

# Affichage du Widget
display(Markdown('## 🎯 Outil 2 : Simulateur de Rendement Locatif (Besoin n°4, 9, 10)'))
display(Markdown('**Instruction pour Hugo :** Ajustez les paramètres selon vos hypothèses (Ville, Prix par Nuit, Taux d\'Occupation Cible) pour simuler votre rentabilité nette.'))
display(interactive_simulation)

## 🎯 Outil 2 : Simulateur de Rendement Locatif (Besoin n°4, 9, 10)

**Instruction pour Hugo :** Ajustez les paramètres selon vos hypothèses (Ville, Prix par Nuit, Taux d'Occupation Cible) pour simuler votre rentabilité nette.

interactive(children=(Dropdown(description='Ville :', options=('ABBEVILLE', 'ABLON-SUR-SEINE', 'ABONDANCE', 'A…